<a href="https://colab.research.google.com/github/LucasTonolli/projeto-ciencia-dados/blob/main/Projeto_Pr%C3%A9_processamento_de_Dados_em_Ci%C3%AAncia_de_Dad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Importação de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Bibliotecas importadas com sucesso.')

Bibliotecas importadas com sucesso.


#Análise inicial dos dados

## Leitura do dataset

In [ ]:
df = pd.read_csv('sample_data/vgsales.csv',quotechar='"',
    encoding='utf-8',
    on_bad_lines='skip')

print("=== DATAFRAME ORIGINAL ===")
print(f'Dataset carregado: {df.shape[0]} linhas e {df.shape[1]} colunas')
print(df.tail(1))

=== DATAFRAME ORIGINAL ===
Dataset carregado: 16598 linhas e 11 colunas
        Rank              Name Platform    Year     Genre Publisher  NA_Sales  \
16597  16600  Spirits & Spells      GBA 2003.00  Platform   Wanadoo      0.01   

       EU_Sales  JP_Sales  Other_Sales  Global_Sales  
16597      0.00      0.00         0.00          0.01  


## Estrutura e tipos de variáveis

In [ ]:
print("\n=== INFORMAÇÕES GERAIS DO DATAFRAME ===")
print(df.info())


=== INFORMAÇÕES GERAIS DO DATAFRAME ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  object 
 2   Platform      16598 non-null  object 
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  object 
 5   Publisher     16540 non-null  object 
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), object(4)
memory usage: 1.4+ MB
None


## Verificação de dados ausentes

In [ ]:
ausentes_nan = df.isnull().sum()

# Contagem de strings 'N/A' e 'Unknown' que o pandas não detecta automaticamente
ausentes_str = pd.Series({
    col: (df[col].astype(str).isin(['N/A', 'Unknown', 'nan'])).sum()
    for col in df.columns
})

df_ausentes = pd.DataFrame({
    'NaN (pandas)': ausentes_nan,
    'Strings inválidas (N/A / Unknown)': ausentes_str,
})
df_ausentes['Total ausentes'] = df_ausentes.sum(axis=1)
df_ausentes['% do total'] = (df_ausentes['Total ausentes'] / len(df) * 100).round(2)

print('=== VALORES AUSENTES POR COLUNA ===')
df_ausentes

=== VALORES AUSENTES POR COLUNA ===


,NaN (pandas),Strings inválidas (N/A / Unknown),Total ausentes,% do total
Rank,0,0,0,0.00
Name,0,0,0,0.00
Platform,0,0,0,0.00
Year,271,271,542,3.27
Genre,0,0,0,0.00
Publisher,58,261,319,1.92
NA_Sales,0,0,0,0.00
EU_Sales,0,0,0,0.00
JP_Sales,0,0,0,0.00
Other_Sales,0,0,0,0.00


## Verificação de inconsistências

### Coluna Publisher: valores problemáticos

In [ ]:
print('=== PUBLISHERS PROBLEMÁTICOS ===')
pub_invalidos = df['Publisher'].astype(str).isin(['N/A', 'Unknown', 'nan', 'NaN'])
print(f'Registros com Publisher inválido: {pub_invalidos.sum()}')

=== PUBLISHERS PROBLEMÁTICOS ===
Registros com Publisher inválido: 261


### Zeros nas colunas de vendas regionais

In [ ]:
colunas_vendas = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']

print('=== REGISTROS COM ZERO EM VENDAS REGIONAIS ===')
zeros = {col: (df[col] == 0).sum() for col in colunas_vendas}
df_zeros = pd.DataFrame({
    'Coluna': zeros.keys(),
    'Qtd zeros': zeros.values(),
    '% do total': [v / len(df) * 100 for v in zeros.values()]
})
print(df_zeros.to_string(index=False))

=== REGISTROS COM ZERO EM VENDAS REGIONAIS ===
     Coluna  Qtd zeros  % do total
   NA_Sales       4499       27.11
   EU_Sales       5730       34.52
   JP_Sales      10455       62.99
Other_Sales       6477       39.02


# Pré-processamento de dados


### Remover Wii Sports - motivo a ser estudado

In [ ]:
df = df[df["Name"] != "Wii Sports"]
print(df.head(10))

    Rank                       Name Platform    Year         Genre Publisher  \
1      2          Super Mario Bros.      NES 1985.00      Platform  Nintendo   
2      3             Mario Kart Wii      Wii 2008.00        Racing  Nintendo   
3      4          Wii Sports Resort      Wii 2009.00        Sports  Nintendo   
4      5   Pokemon Red/Pokemon Blue       GB 1996.00  Role-Playing  Nintendo   
5      6                     Tetris       GB 1989.00        Puzzle  Nintendo   
6      7      New Super Mario Bros.       DS 2006.00      Platform  Nintendo   
7      8                   Wii Play      Wii 2006.00          Misc  Nintendo   
8      9  New Super Mario Bros. Wii      Wii 2009.00      Platform  Nintendo   
9     10                  Duck Hunt      NES 1984.00       Shooter  Nintendo   
10    11                 Nintendogs       DS 2005.00    Simulation  Nintendo   

    NA_Sales  EU_Sales  JP_Sales  Other_Sales  Global_Sales  
1      29.08      3.58      6.81         0.77         40.

### Preencher locais vazios com a média de vendas da publisher por região

In [ ]:
df = df.replace('', pd.NA)

sales_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']

df[sales_cols] = df[sales_cols].apply(pd.to_numeric, errors='coerce')

for col in sales_cols:
    df[col] = df.groupby('Publisher')[col].transform(
        lambda x: x.fillna(x.mean())
    )

print(df)

        Rank                                              Name Platform  \
1          2                                 Super Mario Bros.      NES   
2          3                                    Mario Kart Wii      Wii   
3          4                                 Wii Sports Resort      Wii   
4          5                          Pokemon Red/Pokemon Blue       GB   
5          6                                            Tetris       GB   
...      ...                                               ...      ...   
16593  16596                Woody Woodpecker in Crazy Castle 5      GBA   
16594  16597                     Men in Black II: Alien Escape       GC   
16595  16598  SCORE International Baja 1000: The Official Game      PS2   
16596  16599                                        Know How 2       DS   
16597  16600                                  Spirits & Spells      GBA   

         Year         Genre   Publisher  NA_Sales  EU_Sales  JP_Sales  \
1     1985.00      Platfor

### Normaliza valores de vendas entre 0 e 1

In [ ]:
sales_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']

for col in sales_cols:
    df[col] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())

print(df.head(10))

    Rank                       Name Platform    Year         Genre Publisher  \
1      2          Super Mario Bros.      NES 1985.00      Platform  Nintendo   
2      3             Mario Kart Wii      Wii 2008.00        Racing  Nintendo   
3      4          Wii Sports Resort      Wii 2009.00        Sports  Nintendo   
4      5   Pokemon Red/Pokemon Blue       GB 1996.00  Role-Playing  Nintendo   
5      6                     Tetris       GB 1989.00        Puzzle  Nintendo   
6      7      New Super Mario Bros.       DS 2006.00      Platform  Nintendo   
7      8                   Wii Play      Wii 2006.00          Misc  Nintendo   
8      9  New Super Mario Bros. Wii      Wii 2009.00      Platform  Nintendo   
9     10                  Duck Hunt      NES 1984.00       Shooter  Nintendo   
10    11                 Nintendogs       DS 2005.00    Simulation  Nintendo   

    NA_Sales  EU_Sales  JP_Sales  Other_Sales  Global_Sales  
1       1.00      0.28      0.67         0.07          1.